In [ ]:
# Load 
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
# Add project root to path so we can import from `generation/`
sys.path.insert(0, str(Path.cwd().parent))

import torch
from diffusers import AutoPipelineForText2Image
pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
torch.cuda.empty_cache()
pipe.to("cuda")
print("Loaded pipe!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from LatentPredictionDataset import Latent
def get_latent_lists_from_seeds(seeds: list[int]) -> list[Latent]:
    generators = [torch.Generator(device="cpu").manual_seed(seed) for seed in seeds]
    latents = [
            pipe.prepare_latents(
                batch_size=1,
                num_channels_latents=pipe.unet.config.in_channels,
                height=512,
                width=512,
                dtype=pipe.unet.dtype,
                device="cpu",
                generator=generator,
            )[0] / pipe.scheduler.init_noise_sigma
            for generator in generators
        ]
    return latents

In [16]:
import pandas as pd
from torch.utils.data import DataLoader
from LatentPredictionDataset import create_splits, LatentPredictionDataset

full_dataframe = pd.read_csv("../results/baseline_metrics.csv")
print(f"Loaded {len(full_dataframe)} rows")

# 1. Create the split DataFrames
train_df, val_df = create_splits(full_dataframe)
print(f"Train df Rows: {len(train_df)} | Val dfRows: {len(val_df)}")

# 2. Create two separate Dataset instances
# (They don't know about each other, they just see their own data)
train_dataset = LatentPredictionDataset(
    metrics_df=train_df,
    seeds_to_latent=get_latent_lists_from_seeds,
    alpha_range=(0.1, 0.9)
)

val_dataset = LatentPredictionDataset(
    metrics_df=val_df,
    seeds_to_latent=get_latent_lists_from_seeds,
    alpha_range=(0.1, 0.9)
)

print(f"Train dataset length: {len(train_dataset)} | Val dataset length: {len(val_dataset)}")

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) # No shuffle for Val usually")

Loaded 9600 rows
Total Groups: 600
Train Rows: 8640 | Val Rows: 960
Train df Rows: 8640 | Val dfRows: 960
Train dataset length: 1095 | Val dataset length: 105
